# BigQuery + DVC Data Extraction and EDA

This notebook demonstrates how to pull the three pipeline tables from BigQuery, version them with DVC, and perform exploratory data analysis on `category_tree`, `events`, and `item_properties`.

## Setup

The notebook uses the `BigQueryQuery` helper from `ecommerce_recommender/data_pipeline/bigquery_query.py` to run queries, write CSV exports, and add them to DVC. Make sure your `.venv` is active and DVC is installed.

In [3]:
import sys
from pathlib import Path


def get_repo_root(start_path: Path) -> Path:
    for path in [start_path] + list(start_path.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("Could not locate repository root. Run this notebook inside the repository.")

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = get_repo_root(CURRENT_DIR)
print(f"Repository root found at: {REPO_ROOT}")

sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ecommerce_recommender.data_pipeline.bigquery_query import BigQueryQuery


Repository root found at: /home/dpaula/code/MLENG_FIAP/fase_2


## Configure the Extractor

Set the BigQuery project and dataset values, then create an output folder for the exported CSVs.

In [4]:
PROJECT_ID = "retail-rocket-498618"  # update to your project id
DATASET_ID = "retailrocket"  # update to your dataset id

OUTPUT_DIR = REPO_ROOT / "ecommerce_recommender" / "data" / "raw"
DVC_REPO_PATH = REPO_ROOT

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

query_client = BigQueryQuery(
    project_id=PROJECT_ID,
    dataset_id=DATASET_ID,
    output_dir=OUTPUT_DIR,
    dvc_repo_path=DVC_REPO_PATH,
)


/home/dpaula/code/MLENG_FIAP/fase_2/.venv/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## Extract the Three Pipeline Tables

Export the tables to local CSV files and version them with DVC. If a CSV already exists, the helper will reuse it by default and skip re-downloading unless `force=True` is passed.

In [5]:
category_tree_csv = query_client.extract_table(
    table_name='retail-rocket_category_tree',
    destination_name='category_tree.csv',
)
events_csv = query_client.extract_table(
    table_name='retail-rocket_events',
    destination_name='events.csv',
)
item_properties_csv = query_client.extract_table(
    table_name='retail-rocket_item_properties',
    destination_name='item_properties.csv',
)

category_tree_csv, events_csv, item_properties_csv

Skipping BigQuery export for 'category_tree.csv' because the file already exists. Pass force=True to overwrite and create a new DVC version.
Skipping BigQuery export for 'events.csv' because the file already exists. Pass force=True to overwrite and create a new DVC version.
Skipping BigQuery export for 'item_properties.csv' because the file already exists. Pass force=True to overwrite and create a new DVC version.


(PosixPath('/home/dpaula/code/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/category_tree.csv'),
 PosixPath('/home/dpaula/code/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/events.csv'),
 PosixPath('/home/dpaula/code/MLENG_FIAP/fase_2/ecommerce_recommender/data/raw/item_properties.csv'))

## Load Extracted CSVs

Read the exported files into pandas DataFrames for analysis.

In [6]:
df_category_tree = pd.read_csv(category_tree_csv)
df_events = pd.read_csv(events_csv)
df_item_properties = pd.read_csv(item_properties_csv)

print('Loaded category_tree shape:', df_category_tree.shape)
print('Loaded events shape:', df_events.shape)
print('Loaded item_properties shape:', df_item_properties.shape)

df_category_tree.head(), df_events.head(), df_item_properties.head()


/tmp/ipykernel_25021/2657188350.py:3: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_item_properties = pd.read_csv(item_properties_csv)


Loaded category_tree shape: (1669, 2)
Loaded events shape: (2756101, 5)
Loaded item_properties shape: (20275902, 4)


(   categoryid  parentid
 0         231       NaN
 1         791       NaN
 2        1490       NaN
 3         431       NaN
 4         755       NaN,
        timestamp  visitorid event  itemid  transactionid
 0  1433221955914     483717  view  253185            NaN
 1  1433223291897     794181  view  439202            NaN
 2  1433224554732     453474  view  250696            NaN
 3  1433221377547    1153198  view  388242            NaN
 4  1433224266445     273888  view  205392            NaN,
        timestamp  itemid property                    value
 0  1440298800000   38377      501                   769062
 1  1440298800000  196511      550                   769062
 2  1440903600000  456233      294          1222692 1222090
 3  1440903600000  345779      188  1297729 n1200.000 10317
 4  1440903600000  331187      188  1297729 n1200.000 10317)

## Category Tree EDA

Inspect the hierarchical structure and count categories.

In [9]:
print('Category tree rows:', len(df_category_tree))
print('Unique category IDs:', df_category_tree['categoryid'].nunique())
print('Top parent categories:')
print(df_category_tree['parentid'].value_counts().head(10))

Category tree rows: 1669
Unique category IDs: 1669
Top parent categories:
parentid
250.0     31
362.0     22
1009.0    22
351.0     19
1259.0    18
1687.0    17
312.0     15
945.0     15
893.0     13
1482.0    13
Name: count, dtype: int64


## Events EDA

Analyze event counts, unique users, and event types.

In [12]:
print('Total events:', len(df_events))
print('Unique users:', df_events['visitorid'].nunique())

if 'event' in df_events.columns:
    print('Event type counts:')
    print(df_events['event'].value_counts().head(10))
else:
    print('No event type column in events table.')

Total events: 2756101
Unique users: 1407580
Event type counts:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


## Item Properties EDA

Understand how many unique items and properties exist in the merged properties table.

In [14]:
print('Total item properties rows:', len(df_item_properties))
print('Unique items:', df_item_properties['itemid'].nunique())
print('Sample properties:')
print(df_item_properties.head(10))

Total item properties rows: 20275902
Unique items: 417053
Sample properties:
       timestamp  itemid property  \
0  1440298800000   38377      501   
1  1440298800000  196511      550   
2  1440903600000  456233      294   
3  1440903600000  345779      188   
4  1440903600000  331187      188   
5  1440903600000   49888      188   
6  1441508400000  112680     1080   
7  1441508400000   33199      161   
8  1441508400000   95087      981   
9  1442113200000  457348      225   

                                               value  
0                                             769062  
1                                             769062  
2                                    1222692 1222090  
3                            1297729 n1200.000 10317  
4                            1297729 n1200.000 10317  
5            1297729 n240.000 1178208 n600.000 10317  
6                                   215470 n9048.000  
7                 562911 1084349 46816 592551 353870  
8                   

## Notes

- The exported CSV files are versioned with DVC through `dvc add`.
- If the dataset is large, use filtered queries or `LIMIT` to reduce output size for exploratory analysis.
- Update `PROJECT_ID` and `DATASET_ID` to match your environment before running.